# 07 Email/web features: 25 folds (5x5) for the four sets that matter
Run only after the notebook-06 audit found no leakage (see `docs/PROJECT_LOG.md` 6d). Sets: W2, W2+calendar, W2+E+H, W2+E+H generic; RF and XGBoost with the fixed notebook-04 hyperparameters; no tuning; shared `make_folds` folds, split by user.
The pre-registered verdict from the 5-fold exploration (AMBIGUOUS) is not edited by this notebook. Everything here is descriptive: folds are correlated (the 5 repeats re-use the same users), so stds understate uncertainty.
CV splits by USER, not by time. The calendar week looks easy under user-split CV (no attack before calendar week 23); a time-based holdout is not run here.

In [1]:
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from itgnn.comms_audit import flag_top_q
from itgnn.commslogs import add_comms_features, comms_feature_sets
from itgnn.data import load_config, prepare
from itgnn.evaluation import alert_budget, evaluate_scores, make_folds
from itgnn.weekly_features import FEATURE_SETS, add_history_features, drop_warmup
from itgnn.windows import label_weeks, load_attack_windows, make_weekly_features, week_masks

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
cfg = load_config("configs/config.yaml")
SEED, QS = cfg["seed"], [0.25, 0.5, 1, 2]
LOG = open(os.environ.get("PROBE_LOG", os.devnull), "a")
pd.options.display.float_format = "{:,.3f}".format
pd.options.display.width = 220
pd.options.display.max_columns = 60
pd.options.display.max_rows = 200

d = prepare(cfg)
users, labels = d["users"], d["labels"]
folds = make_folds(users, labels, cfg["cv"]["n_splits"], cfg["cv"]["n_repeats"], SEED)
assert len(folds) == 25
windows = load_attack_windows(cfg["raw_dir"])
ins = pd.read_csv(Path(cfg["raw_dir"]) / "answers" / "insiders.csv")
scen = ins[ins["dataset"].astype(str) == "4.2"].set_index("user")["scenario"]
SCEN = sorted(scen.unique())

full = make_weekly_features(d["logon"], d["device"], d["file"], users)
full["label"] = label_weeks(full, windows)
full["cal_week"] = (full["week_start"] - full["week_start"].min()).dt.days // 7
full = add_history_features(full, d["logon"])
full, dropped = add_comms_features(full, pd.read_parquet("data/processed/email.parquet"), pd.read_parquet("data/processed/http_user_week_domain.parquet"))
weekly = drop_warmup(full); y = weekly["label"]
saved = pd.read_parquet("results/weekly_v2_oof_scores.parquet", columns=["user", "week_start", "model", "repeat", "fold", "score"])
k1 = weekly[["user", "week_start"]].sort_values(["user", "week_start"]).reset_index(drop=True)
k2 = saved[saved["model"] == "RF W2"].drop_duplicates(["user", "week_start"])[["user", "week_start"]].sort_values(["user", "week_start"]).reset_index(drop=True)
pd.testing.assert_frame_equal(k1, k2)
ALL = comms_feature_sets(FEATURE_SETS["W2"])
SETS = {k: ALL[k] for k in ["W2", "W2+calendar", "W2+E+H", "W2+E+H generic"]}
assert np.isfinite(weekly[sorted({c for v in SETS.values() for c in v})].to_numpy(float)).all()
print(f"rows {len(weekly):,}, attack weeks {int(y.sum())}, dropped (events without a Logon row) {dropped}; same row keys as the saved W2 run; 25 folds")

rows 63,238, attack weeks 322, dropped (events without a Logon row) {'email': 0, 'web': 0}; same row keys as the saved W2 run; 25 folds


## Fits: 4 sets x RF/XGBoost x 25 folds = 200 fits

In [2]:
def rf():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
def xgb(ytr):
    return XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, scale_pos_weight=(ytr == 0).sum() / (ytr == 1).sum(),
                         n_jobs=4, random_state=SEED, verbosity=0)
ALGOS = {"RF": lambda ytr: rf(), "XGB": xgb}
t0 = time.time(); metric_rows, oof_parts = [], []
for i, f in enumerate(folds):
    tr, te = week_masks(weekly, f)
    for sname, cols in SETS.items():
        for a, mk in ALGOS.items():
            s = mk(y[tr]).fit(weekly.loc[tr, cols], y[tr]).predict_proba(weekly.loc[te, cols])[:, 1]
            assert np.isfinite(s).all()
            name = f"{a} {sname}"
            metric_rows.append({"model": name, "algo": a, "set": sname, "repeat": f["repeat"], "fold": f["fold"], **evaluate_scores(y[te].to_numpy(), s, k=100)})
            oof_parts.append(pd.DataFrame({"user": weekly.loc[te, "user"].to_numpy(), "week_start": weekly.loc[te, "week_start"].to_numpy(),
                                           "cal_week": weekly.loc[te, "cal_week"].to_numpy(), "repeat": f["repeat"], "fold": f["fold"],
                                           "model": name, "score": s, "label": y[te].to_numpy()}))
    print(f"fold {i + 1}/25 done, elapsed {time.time() - t0:.0f}s", file=LOG, flush=True)
results = pd.DataFrame(metric_rows); oof = pd.concat(oof_parts, ignore_index=True)
oof.to_parquet("results/comms_25fold_oof_scores.parquet", index=False)
print(f"200 fits in {time.time() - t0:.0f}s; saved results/comms_25fold_oof_scores.parquet {oof.shape}")

200 fits in 420s; saved results/comms_25fold_oof_scores.parquet (2529520, 8)


## Reproduction check: W2 over 25 folds must equal the saved step-4 scores

In [3]:
rep = []
for a in ["RF", "XGB"]:
    sv = saved[saved["model"] == f"{a} W2"]; now = oof[oof["model"] == f"{a} W2"]
    m = now.merge(sv, on=["user", "week_start", "repeat", "fold"], suffixes=("", "_saved"))
    assert len(m) == len(now) == len(sv)
    rep.append({"algo": a, "rows matched": len(m), "max |score diff|": (m["score"] - m["score_saved"]).abs().max()})
display(pd.DataFrame(rep))

,algo,rows matched,max |score diff|
0,RF,316190,0.000
1,XGB,316190,0.000


## PR-AUC and paired differences (25 folds)

In [4]:
order = [f"{a} {s}" for a in ["RF", "XGB"] for s in SETS]
S = results.groupby("model")[["pr_auc", "roc_auc", "precision_at_k", "recall_at_k"]].agg(["mean", "std"])
display(pd.DataFrame({m: S[m].apply(lambda r: f"{r['mean']:.3f} ± {r['std']:.3f}", axis=1) for m in ["pr_auc", "roc_auc", "precision_at_k", "recall_at_k"]}).loc[order])
pv = results.assign(idx=results["repeat"] * 5 + results["fold"]).pivot_table(index="idx", columns="model", values="pr_auc")
rows = {}
for a in ["RF", "XGB"]:
    for hi, lo in [("W2+calendar", "W2"), ("W2+E+H", "W2"), ("W2+E+H generic", "W2"), ("W2+E+H", "W2+calendar"), ("W2+E+H generic", "W2+calendar"), ("W2+E+H", "W2+E+H generic")]:
        dl = pv[f"{a} {hi}"] - pv[f"{a} {lo}"]
        per_rep = dl.groupby(np.arange(25) // 5).mean()
        rows[f"{a}: {hi} - {lo}"] = {"mean diff": dl.mean(), "std over 25 folds": dl.std(), "folds positive": f"{int((dl > 0).sum())}/25",
                                    "min repeat mean": per_rep.min(), "max repeat mean": per_rep.max()}
display(pd.DataFrame(rows).T)
print("'repeat mean' = the paired difference averaged over the 5 folds of one repeat; 5 values, one per repeat (they share users, so they are not independent).")

,pr_auc,roc_auc,precision_at_k,recall_at_k
model,,,,
RF W2,0.503 ± 0.071,0.982 ± 0.013,0.354 ± 0.058,0.554 ± 0.072
RF W2+calendar,0.585 ± 0.091,0.989 ± 0.011,0.414 ± 0.074,0.647 ± 0.092
RF W2+E+H,0.737 ± 0.125,0.996 ± 0.004,0.515 ± 0.107,0.800 ± 0.115
RF W2+E+H generic,0.661 ± 0.101,0.986 ± 0.014,0.441 ± 0.078,0.690 ± 0.109
XGB W2,0.458 ± 0.080,0.986 ± 0.012,0.328 ± 0.056,0.513 ± 0.069
XGB W2+calendar,0.555 ± 0.091,0.993 ± 0.005,0.402 ± 0.079,0.626 ± 0.097
XGB W2+E+H,0.867 ± 0.071,0.997 ± 0.004,0.569 ± 0.102,0.883 ± 0.077
XGB W2+E+H generic,0.792 ± 0.086,0.992 ± 0.011,0.527 ± 0.090,0.820 ± 0.083


,mean diff,std over 25 folds,folds positive,min repeat mean,max repeat mean
RF: W2+calendar - W2,0.082,0.046,24/25,0.071,0.091
RF: W2+E+H - W2,0.234,0.111,24/25,0.174,0.281
RF: W2+E+H generic - W2,0.158,0.087,24/25,0.134,0.176
RF: W2+E+H - W2+calendar,0.153,0.109,23/25,0.103,0.191
RF: W2+E+H generic - W2+calendar,0.076,0.103,19/25,0.054,0.098
RF: W2+E+H - W2+E+H generic,0.077,0.117,19/25,0.040,0.110
XGB: W2+calendar - W2,0.097,0.041,25/25,0.092,0.103
XGB: W2+E+H - W2,0.409,0.090,25/25,0.392,0.422
XGB: W2+E+H generic - W2,0.334,0.096,25/25,0.307,0.359
XGB: W2+E+H - W2+calendar,0.313,0.091,25/25,0.300,0.330


'repeat mean' = the paired difference averaged over the 5 folds of one repeat; 5 values, one per repeat (they share users, so they are not independent).


## Alert budget, both models, q = 0.25 / 0.5 / 1 / 2 % (mean ± std over the 5 repeats; caught of 70 / normal users flagged of 930)

In [5]:
scen_of = scen.to_dict()
bud = alert_budget(oof, labels, scen_of, QS, SEED)
bud["attack weeks flagged"] = sum(bud[f"attack weeks s{s}"] for s in SCEN)
bud["precision (weeks)"] = bud["attack weeks flagged"] / bud["weeks flagged"]
agg_ = bud.groupby(["model", "q%"])
fmt = lambda c: agg_[c].agg(lambda s: f"{s.mean():.1f} ± {s.std():.1f}")
tbl = pd.DataFrame({"caught /70": fmt("caught"), "normal flagged": fmt("normal flagged"), "weeks flagged": agg_["weeks flagged"].mean().round(0),
                    "attack weeks flagged": fmt("attack weeks flagged"), "week precision": agg_["precision (weeks)"].mean(),
                    **{f"caught s{s}": fmt(f"caught s{s}") for s in SCEN}})
for a in ["RF", "XGB"]:
    print(f"--- {a}"); display(tbl.loc[[f"{a} {s}" for s in SETS]])
print("eligible insiders per scenario: s1 30, s2 30, s3 10")

--- RF


caught /70 normal flagged  weeks flagged attack weeks flagged  week precision   caught s1   caught s2  caught s3
model             q%                                                                                                                     
RF W2             0.250  51.4 ± 2.3      3.2 ± 0.4        160.000          111.8 ± 2.6           0.697  23.0 ± 1.2  23.4 ± 0.5  5.0 ± 0.7
                  0.500  56.6 ± 1.7      9.2 ± 1.1        319.000          151.0 ± 2.2           0.474  27.0 ± 1.0  24.0 ± 0.0  5.6 ± 0.9
                  1.000  61.2 ± 0.8     18.2 ± 3.6        635.000          194.2 ± 5.9           0.306  28.8 ± 0.4  26.4 ± 0.5  6.0 ± 0.0
                  2.000  65.4 ± 0.9     35.6 ± 4.6      1,267.000          240.2 ± 9.3           0.190  29.6 ± 0.5  29.2 ± 0.4  6.6 ± 0.5
RF W2+calendar    0.250  46.2 ± 1.6      2.4 ± 0.5        160.000          120.8 ± 4.3           0.753  19.0 ± 1.2  23.4 ± 0.5  3.8 ± 0.8
                  0.500  55.2 ± 1.9      7.2 ± 2.2        319.000          171.4 ± 4.0           0.538  26.0 ± 1.0  24.2 ± 0.4  5.0 ± 1.2
                  1.000  61.6 ± 1.5     22.4 ± 4.4        635.000          228.2 ± 5.5           0.360  28.8 ± 0.4  26.8 ± 0.8  6.0 ± 0.7
                  2.000  66.6 ± 0.5     61.8 ± 7.4      1,267.000          283.8 ± 2.2           0.224  30.0 ± 0.0  29.4 ± 0.5  7.2 ± 0.4
RF W2+E+H         0.250  37.8 ± 4.8      0.4 ± 0.9        160.000          137.2 ± 9.2           0.855  15.4 ± 4.4  19.8 ± 1.1  2.6 ± 0.9
                  0.500  53.4 ± 3.4      4.2 ± 2.6        319.000         207.2 ± 12.0           0.650  25.2 ± 2.8  23.4 ± 0.9  4.8 ± 0.8
                  1.000  65.8 ± 0.8     25.0 ± 5.2        635.000          272.6 ± 4.2           0.430  30.0 ± 0.0  27.8 ± 0.8  8.0 ± 1.0
                  2.000  69.4 ± 1.3     71.0 ± 5.8      1,267.000          302.6 ± 7.6           0.239  30.0 ± 0.0  29.6 ± 0.9  9.8 ± 0.4
RF W2+E+H generic 0.250  42.0 ± 3.1      2.0 ± 1.4        160.000          139.2 ± 4.1           0.868  16.2 ± 1.8  20.0 ± 1.2  5.8 ± 0.4
                  0.500  53.6 ± 1.8      5.4 ± 2.3        319.000          190.2 ± 6.1           0.597  24.8 ± 1.3  22.4 ± 0.9  6.4 ± 0.5
                  1.000  60.6 ± 1.3     20.2 ± 4.8        635.000          234.0 ± 4.7           0.369  28.2 ± 1.5  24.6 ± 0.9  7.8 ± 0.4
                  2.000  66.2 ± 1.6     52.0 ± 1.6      1,267.000          262.4 ± 2.4           0.207  29.8 ± 0.4  27.2 ± 1.1  9.2 ± 0.8

--- XGB


caught /70 normal flagged  weeks flagged attack weeks flagged  week precision   caught s1   caught s2  caught s3
model              q%                                                                                                                     
XGB W2             0.250  48.0 ± 3.1      2.8 ± 0.8        160.000           99.4 ± 6.1           0.620  20.2 ± 2.4  24.0 ± 0.7  3.8 ± 0.4
                   0.500  52.6 ± 0.9      5.4 ± 0.5        319.000          138.8 ± 1.1           0.435  23.6 ± 0.5  25.2 ± 0.4  3.8 ± 0.4
                   1.000  56.6 ± 1.3     10.2 ± 1.8        635.000          176.8 ± 6.0           0.279  26.4 ± 1.1  26.2 ± 0.4  4.0 ± 0.0
                   2.000  62.6 ± 1.3     30.2 ± 6.8      1,267.000          246.8 ± 4.5           0.195  28.8 ± 1.1  28.6 ± 0.5  5.2 ± 0.8
XGB W2+calendar    0.250  44.2 ± 1.5      3.0 ± 1.9        160.000          108.4 ± 4.5           0.676  18.0 ± 1.9  24.2 ± 0.4  2.0 ± 0.0
                   0.500  49.8 ± 1.9      6.0 ± 1.2        319.000          156.2 ± 6.1           0.490  22.2 ± 1.1  25.2 ± 1.1  2.4 ± 0.5
                   1.000  58.0 ± 2.0     21.2 ± 6.9        635.000          231.0 ± 2.9           0.364  27.0 ± 1.4  27.2 ± 0.8  3.8 ± 0.8
                   2.000  64.0 ± 1.6     60.6 ± 4.4      1,267.000          290.8 ± 4.4           0.229  29.4 ± 0.5  29.2 ± 0.4  5.4 ± 0.9
XGB W2+E+H         0.250  52.8 ± 3.0      0.2 ± 0.4        160.000          156.2 ± 4.1           0.974  28.6 ± 1.7  22.6 ± 1.3  1.6 ± 0.5
                   0.500  61.6 ± 1.1      6.6 ± 2.1        319.000          246.6 ± 4.3           0.774  30.0 ± 0.0  27.2 ± 0.8  4.4 ± 1.1
                   1.000  67.6 ± 0.9     25.6 ± 3.1        635.000          296.6 ± 2.3           0.467  30.0 ± 0.0  29.4 ± 0.5  8.2 ± 0.8
                   2.000  69.0 ± 0.0     58.6 ± 5.7      1,267.000          312.0 ± 2.6           0.246  30.0 ± 0.0  30.0 ± 0.0  9.0 ± 0.0
XGB W2+E+H generic 0.250  47.0 ± 2.4      1.4 ± 1.1        160.000          148.8 ± 4.7           0.928  17.8 ± 2.2  24.2 ± 1.5  5.0 ± 1.0
                   0.500  58.4 ± 1.1      8.2 ± 2.2        319.000          232.2 ± 6.8           0.728  25.4 ± 1.3  26.8 ± 1.1  6.2 ± 1.3
                   1.000  66.0 ± 1.0     27.2 ± 5.2        635.000          276.4 ± 5.6           0.436  29.6 ± 0.9  28.2 ± 0.4  8.2 ± 1.1
                   2.000  67.6 ± 0.5     61.6 ± 3.1      1,267.000          295.8 ± 4.5           0.233  29.8 ± 0.4  29.0 ± 0.0  8.8 ± 0.4

eligible insiders per scenario: s1 30, s2 30, s3 10


## RF at q = 0.25%: W2+E+H catches fewer distinct insiders than W2 (repeat 0 showed 41 vs 49). Is it concentration?

In [6]:
rows = []
for m in order:
    for r in range(5):
        om = oof[(oof["model"] == m) & (oof["repeat"] == r)]
        fl = flag_top_q(om, 0.25, SEED + r); g = om[fl]; att = g[g["label"] == 1]; per = att.groupby("user").size().sort_values(ascending=False)
        rows.append({"model": m, "repeat": r, "weeks flagged": len(g), "attack weeks flagged": len(att), "distinct insiders": att["user"].nunique(),
                     "max weeks / insider": int(per.max()), "insiders with >=3 flagged weeks": int((per >= 3).sum()), "top-5 insiders' share": per.head(5).sum() / len(att)})
cc = pd.DataFrame(rows).groupby("model").mean(numeric_only=True).drop(columns="repeat").loc[order]
cc["precision"] = cc["attack weeks flagged"] / cc["weeks flagged"]
display(cc)
print("mean over 5 repeats. Precision up while distinct insiders fall = the alert slots pile onto the same insiders' consecutive attack weeks.")

,weeks flagged,attack weeks flagged,distinct insiders,max weeks / insider,insiders with >=3 flagged weeks,top-5 insiders' share,precision
model,,,,,,,
RF W2,160.400,111.800,51.400,5.600,18.800,0.209,0.697
RF W2+calendar,160.400,120.800,46.200,7.800,19.800,0.265,0.753
RF W2+E+H,160.400,137.200,37.800,8.800,16.800,0.303,0.855
RF W2+E+H generic,160.400,139.200,42.000,8.800,18.000,0.288,0.868
XGB W2,160.400,99.400,48.000,7.000,14.000,0.249,0.620
XGB W2+calendar,160.400,108.400,44.200,8.000,16.400,0.287,0.676
XGB W2+E+H,160.400,156.200,52.800,8.200,19.600,0.243,0.974
XGB W2+E+H generic,160.400,148.800,47.000,8.200,20.200,0.260,0.928


mean over 5 repeats. Precision up while distinct insiders fall = the alert slots pile onto the same insiders' consecutive attack weeks.


## Decision-rule quantities on 25 folds (DESCRIPTIVE; the pre-registered verdict from repeat 0 stays AMBIGUOUS)

In [7]:
def dl(a, hi, lo): return pv[f"{a} {hi}"] - pv[f"{a} {lo}"]
q5 = bud[bud["q%"] == 0.5].groupby("model")[["caught", "normal flagged"]].mean()
q1 = bud[bud["q%"] == 1].groupby("model")[["caught", "normal flagged"]].mean()
rows = {
 "RF mean delta W2+E+H - W2 (rule: >= +0.05)": f"{dl('RF', 'W2+E+H', 'W2').mean():.3f}, positive in {int((dl('RF', 'W2+E+H', 'W2') > 0).sum())}/25 folds",
 "XGB mean delta W2+E+H - W2 (rule: >= +0.03)": f"{dl('XGB', 'W2+E+H', 'W2').mean():.3f}, positive in {int((dl('XGB', 'W2+E+H', 'W2') > 0).sum())}/25 folds",
 "calendar control RF / XGB delta vs W2 (rule: < +0.01)": f"{dl('RF', 'W2+calendar', 'W2').mean():.3f} / {dl('XGB', 'W2+calendar', 'W2').mean():.3f}",
 "RF delta vs the stricter baseline W2+calendar": f"{dl('RF', 'W2+E+H', 'W2+calendar').mean():.3f}",
 "XGB delta vs W2+calendar": f"{dl('XGB', 'W2+E+H', 'W2+calendar').mean():.3f}",
 "RF generic delta vs W2 / vs W2+calendar": f"{dl('RF', 'W2+E+H generic', 'W2').mean():.3f} / {dl('RF', 'W2+E+H generic', 'W2+calendar').mean():.3f}",
 "XGB generic delta vs W2 / vs W2+calendar": f"{dl('XGB', 'W2+E+H generic', 'W2').mean():.3f} / {dl('XGB', 'W2+E+H generic', 'W2+calendar').mean():.3f}",
 "RF q=0.5%: caught W2+E+H - W2 / normal flagged diff": f"{q5.loc['RF W2+E+H', 'caught'] - q5.loc['RF W2', 'caught']:+.1f} / {q5.loc['RF W2+E+H', 'normal flagged'] - q5.loc['RF W2', 'normal flagged']:+.1f}",
 "RF q=1% (descriptive)": f"{q1.loc['RF W2+E+H', 'caught'] - q1.loc['RF W2', 'caught']:+.1f} / {q1.loc['RF W2+E+H', 'normal flagged'] - q1.loc['RF W2', 'normal flagged']:+.1f}",
}
display(pd.Series(rows).to_frame("value"))

,value
RF mean delta W2+E+H - W2 (rule: >= +0.05),"0.234, positive in 24/25 folds"
XGB mean delta W2+E+H - W2 (rule: >= +0.03),"0.409, positive in 25/25 folds"
calendar control RF / XGB delta vs W2 (rule: < +0.01),0.082 / 0.097
RF delta vs the stricter baseline W2+calendar,0.153
XGB delta vs W2+calendar,0.313
RF generic delta vs W2 / vs W2+calendar,0.158 / 0.076
XGB generic delta vs W2 / vs W2+calendar,0.334 / 0.237
RF q=0.5%: caught W2+E+H - W2 / normal flagged diff,-3.2 / -5.0
RF q=1% (descriptive),+4.6 / +6.8
